In [114]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

#Chemin vers les données sur ma machine
CHEMIN_INTERIM = '../data/interim'

In [115]:
df_LabelConf = spark.read.parquet(CHEMIN_INTERIM + '/df_LabelConf.parquet')
df_LabelConf.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [116]:
# On filtre les lignes où on a les entrées et la sorties de rendemment (pour avoir un rendement valide)
df_PreReg = (
    df_LabelConf
    .select(  
        
        # Identifiants (pas pour le modèle, mais pratiques pour debug/analyse)
        'CdSta'
        ,'DtPrel'
        
        # Features catégorielles (à encoder ensuite)
        ,'CatSta'
        ,'NoDep'
        ,'MoPrel'
        ,'AnPrel'
        
        # Features numériques (entrée + station)
        ,'A3_DBO5'
        ,'A3_DCO'
        ,'A3_MES'
        ,'CapNomSta'
        
        # Cible
        ,'Rdt_DBO5'
        # ,'Rdt_DCO'
        # ,'Rdt_MES'
    )
    .filter(
        sf.col('Rdt_DBO5').isNotNull()
        & sf.col('A3_DBO5').isNotNull()
        & sf.col('A3_DCO').isNotNull()
        & sf.col('A3_MES').isNotNull()
        & sf.col('CapNomSta').isNotNull()
    )
)
print(df_PreReg.count())
df_PreReg.select('Rdt_DBO5','Rdt_DCO','Rdt_MES').describe().show()


36278
+-------+-----------------+-----------------+-----------------+
|summary|         Rdt_DBO5|          Rdt_DCO|          Rdt_MES|
+-------+-----------------+-----------------+-----------------+
|  count|            36278|            36278|            36277|
|   mean|97.47377028259892|94.61832909340521| 96.5188052183369|
| stddev|6.272522501295374|7.197395987598142|9.529754044493345|
|    min|           -100.0|           -100.0|           -100.0|
|    max|            100.0|            100.0|            100.0|
+-------+-----------------+-----------------+-----------------+



### String Indexing & One-Hot Encoding

In [118]:
col_idx = ['AnPrel', 'MoPrel', 'NoDep','CatSta']

# Indexing 
indexeur = [StringIndexer(inputCol = c, outputCol = c + '_idx',handleInvalid="keep") for c in col_idx]

# OHE
encodeur= [OneHotEncoder(inputCol = c + '_idx', outputCol = c + '_ohe',handleInvalid="keep") for c in col_idx]

# df_PreRegEncod
df_PreRegEncod = Pipeline(stages=[*indexeur,*encodeur]).fit(df_PreReg).transform(df_PreReg)
df_PreRegEncod.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- CatSta: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- Rdt_DBO5: double (nullable = true)
 |-- Rdt_DCO: double (nullable = true)
 |-- Rdt_MES: double (nullable = true)
 |-- AnPrel_idx: double (nullable = false)
 |-- MoPrel_idx: double (nullable = false)
 |-- NoDep_idx: double (nullable = false)
 |-- CatSta_idx: double (nullable = false)
 |-- AnPrel_ohe: vector (nullable = true)
 |-- MoPrel_ohe: vector (nullable = true)
 |-- NoDep_ohe: vector (nullable = true)
 |-- CatSta_ohe: vector (nullable = true)



In [119]:
# Features finales
col_feat = ['AnPrel_ohe', 'MoPrel_ohe', 'NoDep_ohe', 'CatSta_ohe','CapNomSta', 'A3_DBO5', 'A3_DCO', 'A3_MES']

# Assembleur : création de la colonne feature
assembleur = VectorAssembler(inputCols = col_feat, outputCol= 'features')

# df_Reg
df_Reg = (
    Pipeline(stages=[assembleur])
    .fit(df_PreRegEncod)
    .transform(df_PreRegEncod)
    .select(  
        # Identifiants (pas pour le modèle, mais pratiques pour debug/analyse)
        'CdSta'
        ,'DtPrel'
        
        # Features catégorielles (à encoder ensuite)
        ,'CatSta'
        ,'NoDep'
        ,'MoPrel'
        ,'AnPrel'
        
        # Features numériques (entrée + station)
        ,'A3_DBO5'
        ,'A3_DCO'
        ,'A3_MES'
        ,'CapNomSta'
        
        # Cible
        ,'Rdt_DBO5'
        ,'Rdt_DCO'
        ,'Rdt_MES'
        
        # feature
        ,'features'
    ))
df_Reg.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- CatSta: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- Rdt_DBO5: double (nullable = true)
 |-- Rdt_DCO: double (nullable = true)
 |-- Rdt_MES: double (nullable = true)
 |-- features: vector (nullable = true)



In [120]:
# creation de données d'entrainement et test data
df_train, df_test = df_Reg.randomSplit([.8,.2],seed=2026)

In [121]:
# # Fonction pour evaluation des differnt métriques du model de regression
# **metricName one of**
# * rmse : root mean squared error (default)
# * r2 : r^2 metric
# * mae : mean absolute error
# * var : explained variance

def eval_metrics(reg_method, df_predictions):
    print(f'{B}Evalution des métriques {reg_method} :{BE}')
    for metric in ['rmse','r2','mae','var']:
        # reg_method = 'Linear Regression' 
        evaluator = RegressionEvaluator(predictionCol = 'prediction' ,labelCol = 'Rdt_DBO5',metricName = metric)
        eval_value = round(evaluator.evaluate(df_predictions),2)
        print(f'{B}{metric}{BE}:{eval_value}')
    # print(f'\n')

### Linear Regression

In [123]:
# Regression lineaire: parametère par default 
lr = LinearRegression(featuresCol='features',labelCol='Rdt_DBO5')
df_lr_pred_NonParam = lr.fit(df_train).transform(df_test)


# Grille des parametères 

lr_grilPram = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 1.0])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .addGrid(lr.maxIter, [20, 50, 100])
    .build()
)
# Evaluateur du model regression lineaire 
lr_evaluateur = RegressionEvaluator(predictionCol = 'prediction' ,labelCol = 'Rdt_DBO5',metricName = 'rmse')

# Validateur croisé des parameters du model
lr_validateur = CrossValidator(
    estimator = lr
    ,estimatorParamMaps = lr_grilPram
    ,evaluator = lr_evaluateur
    ,numFolds=2
    ,parallelism=4
    ,seed=2026
)

# Modele du Validateur
lr_validateur_model = lr_validateur.fit(df_train) # 3x3x2 = 12 models tested

Meil_regPrama =         lr_validateur_model.bestModel.getRegParam()
Meil_ElasticNetParam =  lr_validateur_model.bestModel.getElasticNetParam()
Meil_maxIter =          lr_validateur_model.bestModel.getMaxIter()

# Prediction sur le df_test
df_lr_pred = lr_validateur_model.bestModel.transform(df_test)


# Evalutation du modèle et Affichage
print(f'{B}Meilleur Parametres du modele:{BE}')
print(f'{B}Meilleur RegPrama :{BE} {Meil_regPrama}')
print(f'{B}Meilleur ElasticNetParam :{BE} {Meil_ElasticNetParam}')
print(f'{B}Meilleur MaxIter :{BE} {Meil_maxIter}\n')

eval_metrics('Linear Regression - Sans Parametrage', df_lr_pred_NonParam)
eval_metrics('Linear Regression - Avec les meilleures paramètres', df_lr_pred)

# conclusion le tuning ne change rien 

Meilleur Parametres du modele:
Meilleur RegPrama : 0.01
Meilleur ElasticNetParam : 0.5
Meilleur MaxIter : 20

Evalution des métriques Linear Regression - Sans Parametrage :
rmse:6.76
r2:0.04
mae:2.02
var:1.64
Evalution des métriques Linear Regression - Avec les meilleures paramètres :
rmse:6.77
r2:0.04
mae:2.03
var:1.56


### GBT Regressor

In [125]:
# Gradient Boosted Tree: parametère par default 
gbt = GBTRegressor(featuresCol='features', labelCol='Rdt_DBO5', seed=2026)

df_gbt_pred_NonParam = gbt.fit(df_train).transform(df_test)


# Grille des parametères 

gbt_grilPram = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [20, 50])
    .addGrid(gbt.maxDepth, [3, 5, 8])
    .addGrid(gbt.stepSize, [.05, .1])
    # .addGrid(gbt.subsamplingRate, [.7, .9, 1])
    .build()
)
# Evaluateur du model regression lineaire 
gbt_evaluateur = RegressionEvaluator(predictionCol = 'prediction' ,labelCol = 'Rdt_DBO5',metricName = 'rmse')

# Validateur croisé des parameters du model
gbt_validateur = CrossValidator(
    estimator = gbt
    ,estimatorParamMaps = gbt_grilPram
    ,evaluator = gbt_evaluateur
    ,numFolds=2
    ,parallelism=4
    ,seed=2026
)

# Modele du Validateur
gbt_validateur_model = gbt_validateur.fit(df_train) # 2x3x2 = 12 models tested * 2 folds

Meil_maxIter =         gbt_validateur_model.bestModel.getMaxIter()
Meil_maxDepth =        gbt_validateur_model.bestModel.getMaxDepth()
Meil_stepSize =        gbt_validateur_model.bestModel.getStepSize()

# Prediction sur le df_test
df_gbt_pred = gbt_validateur_model.bestModel.transform(df_test)


# Evalutation du modèle et Affichage
print(f'{B}Meilleur Parametres du modele:{BE}')
print(f'{B}Meilleur MaxIter :{BE} {Meil_maxIter}')
print(f'{B}Meilleur MaxDepth :{BE} {Meil_maxDepth}')
print(f'{B}Meilleur StepSize :{BE} {Meil_stepSize}\n')

eval_metrics('Gradient Boosted Tree - Sans Parametrage', df_gbt_pred_NonParam)
eval_metrics('Gradient Boosted Tree - Avec les meilleures paramètres', df_gbt_pred)

# conclusion le tuning ne change rien 

Meilleur Parametres du modele:
Meilleur MaxIter : 50
Meilleur MaxDepth : 3
Meilleur StepSize : 0.05

Evalution des métriques Gradient Boosted Tree - Sans Parametrage :
rmse:6.39
r2:0.14
mae:1.55
var:15.7
Evalution des métriques Gradient Boosted Tree - Avec les meilleures paramètres :
rmse:6.13
r2:0.21
mae:1.55
var:9.94


### Random Forest

In [149]:
# Regression lineaire: parametère par default 
rf = RandomForestRegressor(featuresCol='features',labelCol='Rdt_DBO5',seed=2026)

df_rf_pred_NonParam = rf.fit(df_train).transform(df_test)


# Grille des parametères 

rf_grilPram = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 15])
    .addGrid(rf.minInstancesPerNode, [1, 5])
    .build()
)

# Evaluateur du model regression lineaire 
rf_evaluateur = RegressionEvaluator(predictionCol = 'prediction' ,labelCol = 'Rdt_DBO5',metricName = 'rmse')

# Validateur croisé des parameters du model
rf_validateur = CrossValidator(
    estimator = rf
    ,estimatorParamMaps = rf_grilPram
    ,evaluator = rf_evaluateur
    ,numFolds=2
    ,parallelism=4
    ,seed=2026
)

# Modele du Validateur
rf_validateur_model = rf_validateur.fit(df_train) # 3x3x2 = 18 models tested

Meil_numTrees =              rf_validateur_model.bestModel.getNumTrees
Meil_maxDepth =              rf_validateur_model.bestModel.getMaxDepth()
Meil_minInstancesPerNode =   rf_validateur_model.bestModel.getMinInstancesPerNode()

# Prediction sur le df_test
df_rf_pred = rf_validateur_model.bestModel.transform(df_test)


# Evalutation du modèle et Affichage
print(f'{B}Meilleur Parametres du modele:{BE}')
print(f'{B}Meilleur NumTrees :{BE} {Meil_numTrees}')
print(f'{B}Meilleur MaxDepthMeil_maxDepth :{BE} {Meil_maxDepth}')
print(f'{B}Meilleur MinInstancesPerNode :{BE} {Meil_minInstancesPerNode}\n')

eval_metrics('Random Forrest Regression - Sans Parametrage', df_rf_pred_NonParam)
eval_metrics('Random Forrest Regression - Avec les meilleures paramètres', df_rf_pred)

# conclusion le tuning ne change rien 

Meilleur Parametres du modele:
Meilleur NumTrees : 50
Meilleur MaxDepthMeil_maxDepth : 15
Meilleur MinInstancesPerNode : 5

Evalution des métriques Random Forrest Regression - Sans Parametrage :
rmse:6.12
r2:0.21
mae:1.63
var:7.28
Evalution des métriques Random Forrest Regression - Avec les meilleures paramètres :
rmse:5.97
r2:0.25
mae:1.44
var:10.15


In [ ]:
# conclusion Random Forest me donne le meilleur résulltat

In [147]:
# Feature imporantces 

attrs = df_train.schema['features'].metadata['ml_attr']['attrs']

feat_names = [None] * df_train.first()['features'].size
for type_attr in ['numeric', 'binary', 'nominal']:
    if type_attr in attrs:
        for a in attrs[type_attr]:
            feat_names[a['idx']] = a['name']

# Affichage avec importances
import numpy as np
imp_array = rf_validateur_model.bestModel.featureImportances.toArray()

for nom, imp in sorted(zip(feat_names, imp_array), key=lambda x: -x[1])[:15]:
    print(f"{nom:<35} {imp:.4f}")

A3_DBO5                             0.2906
A3_DCO                              0.1538
A3_MES                              0.1287
CapNomSta                           0.1264
NoDep_ohe_77                        0.0245
CatSta_ohe_petite                   0.0239
AnPrel_ohe_2021                     0.0213
MoPrel_ohe_04                       0.0203
MoPrel_ohe_12                       0.0199
MoPrel_ohe_05                       0.0140
NoDep_ohe_27                        0.0134
AnPrel_ohe_2022                     0.0131
MoPrel_ohe_06                       0.0116
MoPrel_ohe_01                       0.0114
NoDep_ohe_02                        0.0104


In [175]:
print(f'-------------------------------------------\n-------------------------------------------')

-------------------------------------------
-------------------------------------------
